# Трансформеры за неделю: подготовка проекта

Это **один проект по классификации тональности**, который будем дополнять по вашим ежедневным заданиям. Здесь только подготовка: реальные данные, проверка качества и постоянное разбиение на выборки. Токенизацию добавим по заданию Дня 1.

Пояснения идут перед каждым блоком кода. Выполняйте ячейки сверху вниз: **Kernel → Restart Kernel and Run All Cells**. Для подготовки достаточно CPU; видеокарта не нужна.

## План на неделю

| День | Что изучаем | Что получаем |
|---|---|---|
| 1 | Токенизация | Токены, числовые ID, маски |
| 2 | Hidden states | Контекстные представления токенов |
| 3 | Attention | Визуализация весов внимания |
| 4 | Baseline | Результат классификации без дообучения трансформера |
| 5 | Fine-tuning | Модель, дообученная на нашей задаче |
| 6 | Сравнение | Метрики на одной отложенной выборке |
| 7 | Анализ и приложение | Разбор ошибок и демо |

Под «без дообучения» в День 4 можно понимать замороженный трансформер и обученный поверх его представлений простой классификатор. Детали уточним по заданию. Случайная классификационная голова сама по себе полезным baseline не является.

## 1. Окружение

Рекомендуется отдельное окружение Python 3.11 или 3.12. В терминале из папки проекта:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
python -m ipykernel install --user --name transformers-week --display-name "Python (transformers-week)"
python -m jupyter lab
```

В Jupyter выберите ядро **Python (transformers-week)**. В Windows активация окружения: `.venv\Scripts\activate`.

Если работаете в Google Colab, загрузите ноутбук и запустите в отдельной ячейке:

```python
%pip install pandas pyarrow scikit-learn
```

В Colab при отсутствии локальных данных ноутбук скачает исходные файлы сам. Интернет нужен только для загрузки и установки пакетов. Библиотеки `torch` и `transformers` установим, когда они понадобятся в День 1.

Следующая ячейка импортирует инструменты и задаёт папки. `SEED` фиксирует случайное разбиение, чтобы все дни работали с одними примерами. `Path.cwd()` — папка, в которой запущено ядро; запускайте Jupyter из папки проекта.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import urllib.request
from importlib.metadata import version

import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

SEED = 42
PROJECT_DIR = Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
DATA_DIR = PROJECT_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Папка проекта:", PROJECT_DIR)
print("Python:", platform.python_version())
print({name: version(name) for name in ("pandas", "pyarrow", "scikit-learn")})

Папка проекта: /Users/alexey_akhmetsin/Documents/ChatGPT/Асинхронность и параллельность
Python: 3.13.5
{'pandas': '2.2.3', 'pyarrow': '19.0.0', 'scikit-learn': '1.6.1'}


## 2. Какие данные используем

[SST-2 от Stanford NLP на Hugging Face](https://huggingface.co/datasets/stanfordnlp/sst2) — английские тексты о фильмах. Метки: **0 — negative**, **1 — positive**. Нейтрального класса здесь нет.

В исходном наборе 67 349 обучающих примеров, 872 размеченных validation и 1 821 test со скрытыми метками (`-1`). Официальный test не загружаем: локально посчитать accuracy на нём нельзя.

В обучающей части встречаются и короткие фрагменты, а не только целые предложения. Это учебный эксперимент: случайное деление таких фрагментов может оставить родственные тексты в train и validation. Поэтому внутренняя validation служит для настройки, а финальную оценку получим на отдельной официальной части.

Ниже зафиксированы ревизия источника и SHA-256 исходных файлов. Это позволяет восстановить именно те данные, на которых начался проект. Если Parquet уже лежат в `data/raw`, повторной загрузки не будет; хеш проверяется при каждом запуске. Никакие ключи или аккаунт Hugging Face не нужны.

In [2]:
DATASET_ID = "stanfordnlp/sst2"
REVISION = '8d51e7e4887a4caaa95b3fbebbf53c0490b58bbb'
EXPECTED_SHA256 = {'train': 'c7921283b75a42e685f50edecb96798607ea0fcbfd0739ee8975f22c12d55f09', 'validation': 'fb00fe008f6828f86ba2beda8415a4cf5da0c884f21c5f238c87131b5aa19529'}
SOURCE_URL = "https://huggingface.co/datasets/stanfordnlp/sst2"

for split in ("train", "validation"):
    path = RAW_DIR / f"{split}.parquet"
    url = (
        f"https://huggingface.co/datasets/{DATASET_ID}/resolve/"
        f"{REVISION}/data/{split}-00000-of-00001.parquet"
    )
    if not path.exists():
        print("Скачиваем:", split)
        try:
            with urllib.request.urlopen(url, timeout=120) as response:
                content = response.read()
        except Exception as exc:
            raise RuntimeError(
                "Не удалось скачать SST-2. Проверьте интернет или скопируйте "
                "папку data/raw из подготовленного проекта рядом с ноутбуком."
            ) from exc
        if hashlib.sha256(content).hexdigest() != EXPECTED_SHA256[split]:
            raise ValueError(f"Загруженный файл {split} не прошёл проверку SHA-256")
        path.write_bytes(content)
    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
    if actual_hash != EXPECTED_SHA256[split]:
        raise ValueError(f"Файл {path} отличается от исходного. Проверьте его перед заменой.")

raw_train = pd.read_parquet(RAW_DIR / "train.parquet")
raw_eval = pd.read_parquet(RAW_DIR / "validation.parquet")
assert len(raw_train) == 67349 and len(raw_eval) == 872
print("Исходный train:", raw_train.shape)
print("Официальная validation:", raw_eval.shape)
display(raw_train.head())

Исходный train: (67349, 3)
Официальная validation: (872, 3)


,idx,sentence,label
0,0,hide new secretions from the parental units,0
1,1,"contains no wit , only labored gags",0
2,2,that loves its characters and communicates som...,1
3,3,remains utterly satisfied to remain the same t...,0
4,4,on the worst revenge-of-the-nerds clichés the ...,0


## 3. Приводим к формату `text,label`

`rename` переименовывает исходную колонку `sentence` в `text`, а выбор колонок убирает служебный индекс `idx`. Метка — целевой ответ, который позже будет предсказывать классификатор.

Для поиска повторов строим отдельный ключ: нижний регистр и одинаковые пробелы. Сам текст оставляем без такой нормализации, убрав лишь пробелы по краям. Пустые тексты исключаем. Если одинаковый ключ имеет разные метки, удаляем всю спорную группу; среди остальных точных повторов сохраняем одну строку. Это не поиск смысловых дублей.

In [3]:
def prepare_frame(raw):
    frame = raw.rename(columns={"sentence": "text"})[["text", "label"]].copy()
    assert frame["label"].isin([0, 1]).all(), "Неожиданные метки"
    frame = frame.dropna(subset=["text", "label"])
    frame["text"] = frame["text"].str.strip()
    frame = frame.loc[frame["text"].str.len() > 0].copy()
    frame["label"] = frame["label"].astype("int64")
    frame["_key"] = frame["text"].str.lower().str.replace(r"\s+", " ", regex=True)
    label_counts = frame.groupby("_key")["label"].nunique()
    ambiguous_keys = label_counts[label_counts > 1].index
    frame = frame.loc[~frame["_key"].isin(ambiguous_keys)]
    return frame.drop_duplicates("_key").reset_index(drop=True)

train_pool = prepare_frame(raw_train)
# Официальная validation становится нашим финальным test.
test_df = prepare_frame(raw_eval)

# Удаляем из обучающего пула точные совпадения с будущим test.
train_pool = train_pool.loc[~train_pool["_key"].isin(test_df["_key"])].reset_index(drop=True)

print("Обучающий пул после очистки:", len(train_pool))
print("Финальная отложенная выборка после очистки:", len(test_df))
display(train_pool[["text", "label"]].head())

Обучающий пул после очистки: 66973
Финальная отложенная выборка после очистки: 872


,text,label
0,hide new secretions from the parental units,0
1,"contains no wit , only labored gags",0
2,that loves its characters and communicates som...,1
3,remains utterly satisfied to remain the same t...,0
4,on the worst revenge-of-the-nerds clichés the ...,0


## 4. Закрепляем выборки на всю неделю

Для быстрых учебных запусков берём **6 000** примеров из очищенного train: **5 000** для обучения и **1 000** для настройки. Остальной обучающий пул сохраняем отдельно, чтобы при необходимости расширить эксперимент позже.

`train_test_split` здесь выбирает случайные строки. `stratify` сохраняет примерно ту же долю каждого класса, а `random_state` делает выбор воспроизводимым. Распределение классов не обязательно 50/50.

- **train**: обучение классификатора и fine-tuning.
- **validation**: выбор настроек и модели.
- **test**: только итоговое сравнение в День 6. До этого не подбираем по нему настройки.

Мы меняем назначение официальной validation и явно фиксируем это в описании. Итоговые метрики относятся к нашему учебному протоколу, а не к официальному скрытому test SST-2.

In [4]:
TRAIN_SIZE = 5000
VALIDATION_SIZE = 1000

study_df, _ = train_test_split(
    train_pool,
    train_size=TRAIN_SIZE + VALIDATION_SIZE,
    stratify=train_pool["label"],
    random_state=SEED,
)
train_df, validation_df = train_test_split(
    study_df,
    train_size=TRAIN_SIZE,
    test_size=VALIDATION_SIZE,
    stratify=study_df["label"],
    random_state=SEED,
)

frames = {
    "train": train_df.reset_index(drop=True),
    "validation": validation_df.reset_index(drop=True),
    "test": test_df.reset_index(drop=True),
}

# Проверяем отсутствие точных пересечений между выборками.
for name, frame in frames.items():
    assert frame["_key"].is_unique, f"Повторы внутри {name}"
    assert frame[["text", "label"]].notna().all().all()
    assert frame["label"].isin([0, 1]).all()
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert set(frames[left]["_key"]).isdisjoint(frames[right]["_key"]), (left, right)

summary = pd.DataFrame({
    name: {
        "rows": len(frame),
        "negative": int((frame["label"] == 0).sum()),
        "positive": int((frame["label"] == 1).sum()),
    }
    for name, frame in frames.items()
}).T
# Только агрегаты; тексты test пока не изучаем.
display(summary)

,rows,negative,positive
train,5000,2213,2787
validation,1000,443,557
test,872,428,444


## 5. Сохраняем CSV и описание эксперимента

Во всех рабочих CSV ровно две колонки: `text,label`. `index=False` не позволяет записать индекс pandas как лишнюю колонку. UTF-8 сохраняет текст без потери символов.

`manifest.json` хранит источник, правила разбиения, размеры, версии библиотек и хеши. Повторный запуск в том же окружении воспроизводит файлы. Если намеренно измените размеры или seed, это уже другое разбиение: будущие сравнения нужно будет пересчитать.

In [5]:
csv_hashes = {}
for name, frame in frames.items():
    path = DATA_DIR / f"{name}.csv"
    frame[["text", "label"]].to_csv(path, index=False, encoding="utf-8")
    csv_hashes[name] = hashlib.sha256(path.read_bytes()).hexdigest()

train_pool[["text", "label"]].to_csv(
    DATA_DIR / "train_pool.csv", index=False, encoding="utf-8"
)
manifest = {
    "dataset": DATASET_ID,
    "source": SOURCE_URL,
    "revision": REVISION,
    "seed": SEED,
    "labels": {"0": "negative", "1": "positive"},
    "protocol": {
        "train": "5000 stratified examples from cleaned official train",
        "validation": "1000 disjoint stratified examples from cleaned official train",
        "test": "cleaned official validation; official test is unused",
        "deduplication": "lowercase + collapsed whitespace; discard conflicting labels",
        "limitation": "Related phrases may remain across train and validation",
    },
    "sizes": {name: len(frame) for name, frame in frames.items()},
    "source_sha256": EXPECTED_SHA256,
    "csv_sha256": csv_hashes,
    "versions": {name: version(name) for name in ("pandas", "pyarrow", "scikit-learn")},
    "python": platform.python_version(),
}
(DATA_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print("CSV сохранены в:", DATA_DIR)
display(pd.read_csv(DATA_DIR / "train.csv").head())

CSV сохранены в: /Users/alexey_akhmetsin/Documents/ChatGPT/Асинхронность и параллельность/data/processed


,text,label
0,with the best of them,1
1,lets you brush up against the humanity of a ps...,1
2,before indigestion sets,0
3,spectacle and pacing,1
4,unforgettable,1


## 6. Первый взгляд на обучающие тексты

Читаем только train: смотрим несколько примеров каждого класса и длины текстов. Здесь длина в **символах**, а не в токенах. Отличие увидим в День 1.

Не удаляем стоп-слова, пунктуацию и отрицания: например, `not good` и `good` имеют разный смысл. Не лемматизируем тексты заранее — используем токенизатор выбранной модели, когда дойдём до него.

In [6]:
train = pd.read_csv(DATA_DIR / "train.csv")
for label, title in [(0, "negative"), (1, "positive")]:
    print(f"\nПримеры {title}:")
    display(train.loc[train["label"] == label].sample(3, random_state=SEED))

print("Длины обучающих текстов в символах:")
display(train["text"].str.len().describe().to_frame("characters"))
print("Готово: данные подготовлены. Следующий шаг — задание Дня 1.")


Примеры negative:


,text,label
1950,goes on for too long and bogs down in a surfei...,0
2928,understatement,0
3275,like a bad blend of an overripe episode of tv ...,0



Примеры positive:


,text,label
3756,famed charisma,1
4970,is impossible not to be awed by the power and ...,1
2634,"is an intelligent , realistic portrayal of tes...",1


Длины обучающих текстов в символах:


,characters
count,5000.000000
mean,52.246800
std,43.377139
min,1.000000
25%,20.000000
50%,37.000000
75%,72.000000
max,258.000000


Готово: данные подготовлены. Следующий шаг — задание Дня 1.


## Что должно быть понятно перед Днём 1

1. `text` — вход модели, `label` — правильный класс.
2. CSV хранит таблицу; токенизатор позже превратит строки в числовой вход трансформера.
3. Train обучает, validation помогает выбрать настройки, test нужен для итоговой оценки.
4. Точные дубли проверены, но смысловые совпадения и родственные фразы этим не исключаются.

## Файлы проекта

```text
00_transformers_overall.ipynb
requirements.txt
README.md
data/
  raw/
    train.parquet
    validation.parquet
    source_metadata.json
  processed/
    train.csv          # 5 000 примеров
    validation.csv     # 1 000 примеров
    test.csv           # очищенная официальная validation
    train_pool.csv     # весь очищенный обучающий пул, включая train и validation
    manifest.json
```

`train_pool.csv` не является дополнительной независимой выборкой. Не объединяйте его с train для обучения, продолжая оценивать на текущей validation: она уже входит в этот пул.

Источники: [карточка SST-2](https://huggingface.co/datasets/stanfordnlp/sst2), [оригинальная страница Stanford Sentiment Treebank](https://nlp.stanford.edu/sentiment/). На Hugging Face лицензия обозначена как `unknown`; для распространения датасета за пределами личного учебного проекта нужно отдельно уточнить условия исходного корпуса.